# Моделирование параметров прибытия подразделений пожарной охраны

## Общие сведения

Моделирование параметров прибытия подразделений пожарной охраны - одна из основных частей работы с библиотекой **Генезис**, в которой при помощи расчетных алгоритмов и с использованием исходных данных происходит моделирование параметров прибытия подразделений пожарной охраны.

В наиболее общем виде структуру моделирования можно представить следующим образом:

```{mermaid}
flowchart LR
  classDef class_a stroke:#ff2d00, color:#ff2d00
  classDef class_b stroke:#0004ff, color:#0004ff
  classDef class_k stroke:#000000

  source["**ИСХОДНЫЕ ДАННЫЕ**"]

  algs["**РАСЧЕТНЫЕ АЛГОРИТМЫ**
        ---
        Кратчайший путь
        ---
        Модель оценки t_сл"]


  model("**МОДЕЛИРОВАНИЕ
          ПАРАМЕТРОВ
          ПРИБЫТИЯ**")
  
  result("**РЕЗУЛЬТАТЫ**")

  algs --> model
  source --> model
  model --> result

```

При конструировании решения пользователю, исходя из поставленной задачи расчета, необходимо: 

1. Указать:
  - алгоритм расчета кратчайшего пути
  - алгоритм расчета времени следования ПА
2. Выбрать исходные данные которые следует использовать:
  - ГДС *(всегда)*
  - дислокация ПСЧ
  - расчетная область
  - скоростной режим *(всегда)*
3. Выбрать алгоритм моделирования

### Алгоритмы моделирования {.unnumbered}

В **Генезис** реализованы следующие алгоритмы:

- `genesis.states.FirstArrivalUnitState`: алгоритм расчета на основе определения времени прибытия от каждой указанной ПСЧ до каждого из узлов ГДС по кратчайшему маршруту (прямая задача)
- `genesis.states.ArrivalTimeMatrixState`: алгоритм расчета на основе заранее рассчитанной матрицы прибытия к каждому из узлов ГДС (обратная задача)

#### Синтаксис genesis.states.FirstArrivalUnitState {.unnumbered}

АРГУМЕНТЫ ИНИЦИАЛИЗАЦИИ

In [ ]:
FirstArrivalUnitState(
    state_algorithm = MSF,
    weight          = 'travel_time',
    delay           = DELAY_TIME)

* `state_algorithm` -- Функция расчета кратчайших путей от множества источников. По умолчанию передается реализация алгоритма Дейкстры `multi_source_dijkstra` из библиотеки `networkx`, указанная в `genesis.swiss_knife.MSF`
* `weight` -- Алгоритм расчета времени следования ПА. По умолчанию - `"travel_time"`
* `delay` -- Задержка. Например на обслуживание вызова на пожар. По умолчанию указана в `genesis.swiss_knife.DELAY_TIME` и равна 1 мин.

АРГУМЕНТЫ ПРИМЕНЕНИЯ

In [ ]:
FirstArrivalUnitState()(
    env,
    points,
    area   = None
    )

* `env` (nx.Graph) -- Граф улично-дорожной сети.
* `points` (dict) -- Стартовые узлы. Словарь вида dict(int:str), где ключ - идентификатор узла, значение - его наименование. Может использоваться для указания узлов в которых расположены пожарные подразделения: {1234:'ПСЧ-1'}
* `area` (pd.Series) -- Маска узлов ГДС. Значениями True отмечены узлы ГДС для которых требуется вернуть результат расчета. Если не указана, расчет производится для всех узлов ГДС.
        

ВОЗВРАЩАЕТ

times, nearest -> tuple[Series[float], Series[str] | Series]. 
* times - Время прибытия первого подразделения в каждый из узлов ГДС. 
* nearest - Соответствие узлов первому прибывающему подразделению.

#### Синтаксис genesis.states.ArrivalTimeMatrixState {.unnumbered}

АРГУМЕНТЫ ИНИЦИАЛИЗАЦИИ

In [ ]:
ArrivalTimeMatrixState(
    matrix,
    state_algorithm = np.min,
    delay           = 0
    )

* `matrix` (pd.DataFrame) -- Матрица времен прибытия, отражающая время прибытия к каждому из объектов из каждого из узлов.
* `state_algorithm` -- Функция оценки времени прибытия из всех возможных стартовых узлов `points`. По умолчанию - `np.min`, что отражает время прибытия первого подразделения.
* `delay` -- Задержка в расчете. Например на обслуживание вызова на пожар. **Предполагается, что уже учтена в `matrix`, поэтому не рекомендуется использовать!**

ВОЗВРАЩАЕТ

times, nearest -> tuple[Series[float], Series[str] | Series]. 
* times - Время прибытия первого подразделения в каждый из узлов ГДС. 
* nearest - Соответствие узлов первому прибывающему подразделению.

АРГУМЕНТЫ ПРИМЕНЕНИЯ

In [ ]:
ArrivalTimeMatrixState(
    env    = None,
    points = None,
    area   = None
    )

* `env` --  Не используется.
* `points` (dict) -- Стартовые узлы. Словарь вида dict(int:str), где ключ - идентификатор узла, значение - его наименование. Может использоваться для указания узлов в которых расположены пожарные подразделения: {1234:'ПСЧ-1'}
* `area` (pd.Series) -- Маска узлов ГДС. Значениями True отмечены узлы ГДС для которых требуется вернуть результат расчета. Если не указана, расчет производится для всех узлов ГДС.

#### Синтаксис genesis.states.get_atm {.unnumbered}

Функция предназначена для расчета матрицы прибытия, используемой в `ArrivalTimeMatrixState`.

АРГУМЕНТЫ ПРИМЕНЕНИЯ

In [ ]:
get_atm(G,
            data: pd.DataFrame    = None,
            data_sample_size: int = None,
            data_node_field: str  = 'node',
            weight: str           = 'travel_time',
            cutoff: float         = None,
            delay: float          = DELAY_TIME,
            target_set: set       = None,
            )

* `G`: nx.MultiDiGraph
    Граф сети городского пожарного хозяйства
* `data`: pd.DataFrame = None
    Данные для расчета. Может быть указан набор данных для которых следует рассчитать,
    например, здания, или набор узлов графа.
    По умолчанию используются все узлы графа
* `data_sample_size`: int = None
    Размер выборки данных для расчета. Количество записей
    data ,которые будут учтены при расчете.
* `data_node_field`: 
    Поле в котором хранится индекс узла, для которого производится расчет.
* `weight`:str или callable  = "travel_time"
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.
* `delay`: float, optional = None
    Задержка в расчете. Например на обслуживание вызова на пожар.
    По умолчанию указана в swiss_knife.DELAY_TIME
* `target_set`: set = None
    Целевой сет узлов графа, которые рассматриваются в качестве потенциальных мест размещения.

:::{.callout-note}

#### Прим.

Помните, что аргументы имеющие значение "по умолчанию", нет необходимости повторно указывать явным образом.

Поэтому в большинстве случаев достаточно использовать синтаксис, например:

    FirstArrivalUnitState()

вместо:

    FirstArrivalUnitState(state_algorithm = MSF,
                weight = 'travel_time',
                delay = DELAY_TIME,
                **kwargs)

:::

### Алгоритмы расчета кратчайшего пути

По умолчанию в **Генезис** используется алгоритм Дейкстры в его реализации для расчета вех маршрутов от нескольких стартовых точек [nx.multi_source_dijkstra](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.multi_source_dijkstra.html).

Данный алгоритм устано

In [ ]:
import osmnx as ox
import geopandas as gpd

## Расчет состояния

```{mermaid}
flowchart LR
  classDef class_a stroke:#ff2d00, color:#ff2d00
  classDef class_b stroke:#0004ff, color:#0004ff
  classDef class_k stroke:#000000

  algs["**РАСЧЕТНЫЕ АЛГОРИТМЫ**
        ---
        Кратчайший путь:
        nx.multi_source_dijkstra
        ---
        Модель оценки t_пр:
        travel_time"]:::class_a 

  source["**ИСХОДНЫЕ ДАННЫЕ**:
         ---
         - Граф улично-дорожной сети
         - Размещение подразделений ПО
         - Скорости следования ПА"]

  model("**МОДЕЛИРОВАНИЕ
          ПАРАМЕТРОВ
          ПРИБЫТИЯ**"):::class_b
  
  result("**РЕЗУЛЬТАТЫ**
          ---
          t_пр")

  algs --> model
  source --> model
  model --> result

```

In [ ]:
from genesis.states import FirstArrivalUnitState




## Загрузка исходных данных

In [ ]:
# Загрузка исходных данных
## Загрузка ГДС
G = ox.load_graphml('data/kemerovo/gds.ml')

## Загрузка зданий

## Загрузка пожарных подразделений
units = gpd.read_file('data/kemerovo/fire_units.gpkg')












In [ ]:
## Установка скоростного режима

In [ ]:
# Выбор расчетных алгоритмов



# Моделирование параметров прибытия
FirstArrivalUnitState()(env=G, points=units)


In [ ]:
# Анализ

# Печать результатов

# Печать карты